# Deep Learning Practical Lab
## Keras Representation → Architecture Comparison → Model Optimization

### Goal
You will:
1. Learn how a neural network is represented in TensorFlow/Keras.
2. Choose **one classification dataset** and build shallow, medium, and deep networks on the same data.
3. Compare their learning curves and generalization.
4. Select one model and improve it using optimization / regularization techniques.
5. Build a final optimized model and justify your decisions with evidence.

> **Fair comparison rule:** Keep the dataset split and preprocessing fixed across experiments.

# Part 1 — Neural Networks in TensorFlow/Keras

TensorFlow provides the deep-learning engine. Keras is the high-level API used to define and train models.

The standard workflow is:

**Define → Compile → Fit → Evaluate → Predict**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


## 1.1 Define the Architecture

Example:

```python
model = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
```

How to read it:

- `Input(shape=(10,))` → 10 input features
- `Dense(64)` → hidden layer with 64 neurons
- `Dense(32)` → second hidden layer with 32 neurons
- `Dense(1, sigmoid)` → binary-classification output

Inside each Dense neuron:

\[
z = w^T x + b
\]

then:

\[
a = g(z)
\]

### Vocabulary

| Concept | Meaning |
|---|---|
| Depth | Number of hidden layers |
| Width | Number of neurons in a layer |
| Parameters | Weights and biases learned from data |
| Hyperparameters | Learning rate, batch size, layers, neurons, dropout, etc. |

## 1.2 Output Layer and Loss

### Binary Classification
```python
layers.Dense(1, activation="sigmoid")
loss="binary_crossentropy"
```

### Multiclass Classification
```python
layers.Dense(n_classes, activation="softmax")
loss="sparse_categorical_crossentropy"
```

Use the multiclass option when the target contains integer class labels such as `0, 1, 2, ...`.

## 1.3 Compile

```python
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
```

- **Loss** → what the model minimizes
- **Optimizer** → how gradients are used to update weights
- **Learning rate** → size of each update
- **Metric** → how we monitor performance

## 1.4 Fit

```python
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)
```

During training:

**Forward Pass → Loss → Backpropagation → Gradients → Weight Update**

`history.history` stores training and validation loss/accuracy across epochs.

## 1.5 Evaluate and Inspect

```python
model.summary()
model.count_params()

model.evaluate(X_test, y_test)
model.predict(X_new)
```

`summary()` shows the architecture and parameter count.  
`evaluate()` is used for final performance on unseen test data.

# Part 2 — Task A: Choose and Prepare One Dataset

Choose **one classification dataset** and use it for the entire lab.

You may use Kaggle, UCI, Scikit-learn, or another instructor-approved public dataset.

### Required dataset description
Write:

1. Dataset name
2. What one row represents
3. Target variable
4. Binary or multiclass
5. Number of rows
6. Number of input features
7. Class distribution
8. Required preprocessing

In [2]:
# TODO: Load your dataset.

df = None

# Examples:
# df = pd.read_csv("data.csv")
# display(df.head())

In [3]:
# TODO: Inspect the dataset.

# print(df.shape)
# display(df.head())
# print(df.dtypes)
# print(df.isna().sum())
# print(df.duplicated().sum())
# print(df["TARGET"].value_counts())
# print(df["TARGET"].value_counts(normalize=True))

In [4]:
# TODO: Define input features X and target y.

X = None
y = None

# Example:
# X = df.drop(columns=["TARGET"])
# y = df["TARGET"]

## Preprocessing Rules

Perform the preprocessing required by your data:

- Handle missing values
- Encode categorical features
- Correct data types
- Scale numerical features when appropriate

> **Avoid data leakage:** fit preprocessing objects such as `StandardScaler` on training data only.

In [5]:
# TODO: Create fixed Train / Validation / Test splits.
# Suggested: 70% / 15% / 15%
#
# X_train, X_temp, y_train, y_temp = train_test_split(
#     X, y, test_size=0.30, random_state=SEED, stratify=y
# )
#
# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp, test_size=0.50,
#     random_state=SEED, stratify=y_temp
# )

In [6]:
# TODO: Scale features if appropriate.
#
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_val = scaler.transform(X_val)
# X_test = scaler.transform(X_test)

In [7]:
# TODO: Determine output configuration after preparing y_train.
#
# input_dim = X_train.shape[1]
# n_classes = len(np.unique(y_train))
#
# if n_classes == 2:
#     output_units = 1
#     output_activation = "sigmoid"
#     loss_function = "binary_crossentropy"
# else:
#     output_units = n_classes
#     output_activation = "softmax"
#     loss_function = "sparse_categorical_crossentropy"
#
# print(input_dim, n_classes, output_activation, loss_function)

# Part 3 — Learning-Curve Helper

Use the same plotting function for every experiment.

In [8]:
def plot_learning_curves(history, title):
    h = history.history

    plt.figure(figsize=(7, 4))
    plt.plot(h["loss"], label="Train Loss")
    plt.plot(h["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " — Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    if "accuracy" in h:
        plt.figure(figsize=(7, 4))
        plt.plot(h["accuracy"], label="Train Accuracy")
        plt.plot(h["val_accuracy"], label="Validation Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title(title + " — Accuracy")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

# Part 4 — Task B: Compare Network Depth

Build **three models** using the same dataset:

### M1 — Shallow
**1 hidden layer**

### M2 — Medium
**3 hidden layers**

### M3 — Deep
**5 or more hidden layers**

For this stage, keep the following fixed:

- Dataset split
- Preprocessing
- Optimizer
- Learning rate
- Batch size
- Maximum epochs

Do **not** use Dropout or BatchNorm yet.

The main variable being tested is **network architecture / depth**.

## Architecture Plan

Complete before training:

| Model | Hidden Layers | Neurons per Layer | Activation | Parameters |
|---|---:|---|---|---:|
| Shallow | | | | |
| Medium | | | | |
| Deep | | | | |

### Prediction
Before running the models, write what you expect to happen as depth increases.

## Task B1 — Shallow Network

In [9]:
def build_shallow_model(input_dim, output_units, output_activation):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # TODO: ONE hidden Dense layer

        # TODO: Output layer
    ])
    return model

In [10]:
# TODO: Build, compile, summarize, train, plot, and evaluate M1.
#
# m1 = build_shallow_model(...)
# m1.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=0.001),
#     loss=loss_function,
#     metrics=["accuracy"]
# )
# m1.summary()
#
# h1 = m1.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=50,
#     batch_size=32,
#     verbose=1
# )
#
# plot_learning_curves(h1, "M1 Shallow")
# m1.evaluate(X_test, y_test)

### M1 Analysis
- Did training loss decrease?
- Did validation loss decrease?
- Is there a large train/validation gap?
- Does the model underfit, overfit, or generalize reasonably well?

## Task B2 — Medium Network

In [11]:
def build_medium_model(input_dim, output_units, output_activation):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # TODO: THREE hidden Dense layers

        # TODO: Output layer
    ])
    return model

In [12]:
# TODO: Train M2 using the SAME optimizer, LR, batch size,
# epochs, data split, and preprocessing used for M1.

### M2 Analysis
Compare with M1:

- Did training performance improve?
- Did validation performance improve?
- Did the generalization gap change?
- Was the extra complexity useful?

## Task B3 — Deep Network

In [13]:
def build_deep_model(input_dim, output_units, output_activation):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # TODO: AT LEAST FIVE hidden Dense layers
        # You may gradually reduce width:
        # 256 -> 128 -> 64 -> 32 -> 16

        # TODO: Output layer
    ])
    return model

In [14]:
# TODO: Train M3 with the SAME training configuration.
# Do not add regularization yet.

### M3 Analysis

- Did deeper achieve lower training loss?
- Did validation improve too?
- Is there evidence of overfitting?
- Around which epoch did validation stop improving?
- Did deeper automatically mean better?

# Part 5 — Architecture Comparison

Create a comparison table:

| Model | Hidden Layers | Parameters | Best Val Accuracy | Min Val Loss | Test Accuracy | Interpretation |
|---|---:|---:|---:|---:|---:|---|
| M1 Shallow | | | | | | |
| M2 Medium | | | | | | |
| M3 Deep | | | | | | |

### Required conclusion

Explain:

1. Which model learned the training data best?
2. Which model generalized best?
3. Which model showed the largest train/validation gap?
4. Was the deepest model always the best?
5. Which architecture will you take to the optimization stage, and why?

In [15]:
architecture_results = pd.DataFrame(columns=[
    "Model", "Hidden Layers", "Parameters",
    "Best Val Accuracy", "Min Val Loss",
    "Test Accuracy", "Interpretation"
])

architecture_results

,Model,Hidden Layers,Parameters,Best Val Accuracy,Min Val Loss,Test Accuracy,Interpretation


# Part 6 — Task C: Optimize One Selected Model

Select **one architecture** from Part 4.

Use the same:
- Data
- Train / validation / test split
- Preprocessing
- Output layer
- Evaluation metric

For every optimization experiment:

**Hypothesis → Change ONE setting → Train fresh model → Plot curves → Compare → Conclude**

## C0 — Baseline

Rebuild the selected architecture using its original settings.

Record:

- Optimizer
- Learning rate
- Batch size
- Epochs
- Parameter count
- Best validation accuracy
- Minimum validation loss
- Test accuracy
- Learning-curve diagnosis

In [16]:
# TODO: Rebuild your selected model as the optimization baseline.

## C1 — Learning Rate Tuning

Keep the architecture fixed.

Test:

```python
1e-2
1e-3
1e-4
```

For every learning rate, **rebuild the model from fresh weights**.

Compare:
- Training speed
- Stability
- Validation performance

In [17]:
learning_rates = [1e-2, 1e-3, 1e-4]

# TODO:
# For every LR:
# 1. Build a fresh copy of the same architecture
# 2. Compile with Adam and that LR
# 3. Train on the same split
# 4. Record best validation result
# 5. Compare learning curves

## C2 — Batch Size Tuning

Using the best learning rate from C1, test:

```python
16
32
64
```

Keep everything else fixed.

Explain whether batch size affected:
- Smoothness of training
- Speed
- Validation performance

In [18]:
batch_sizes = [16, 32, 64]

# TODO: Train fresh models and compare.

## C3 — Early Stopping

Use:

```python
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
```

Train with a high maximum such as `epochs=100`.

Analyze:
- When did training stop?
- Was it before epoch 100?
- Did it stop near the best validation point?

In [19]:
# TODO:
# early_stop = keras.callbacks.EarlyStopping(
#     monitor="val_loss",
#     patience=5,
#     restore_best_weights=True
# )
#
# history = model.fit(
#     ...,
#     epochs=100,
#     callbacks=[early_stop]
# )

## C4 — Dropout

Add Dropout to the same architecture.

Suggested experiments:

```python
Dropout(0.2)
Dropout(0.3)
Dropout(0.5)
```

Compare against the baseline.

Questions:
- Did training accuracy decrease?
- Did the train/validation gap become smaller?
- Did validation or test performance improve?

In [20]:
def build_dropout_model(input_dim, output_units, output_activation, dropout_rate=0.3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # TODO: Recreate your selected architecture
        # Add Dropout after selected hidden layers

        # TODO: Output
    ])
    return model

## C5 — Batch Normalization

Add:

```python
layers.BatchNormalization()
```

after selected Dense layers.

Compare with the baseline:

- Did training become more stable?
- Did it converge faster?
- Did validation performance improve?

In [21]:
def build_batchnorm_model(input_dim, output_units, output_activation):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # TODO: Same architecture + BatchNormalization

        # TODO: Output
    ])
    return model

## C6 — Optional L2 Regularization

Example:

```python
from tensorflow.keras.regularizers import l2

layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=l2(0.001)
)
```

Test whether L2 reduces overfitting on your selected model.

In [22]:
# TODO: Optional L2 experiment.

# Part 7 — Final Optimized Model

Build one final model using **only the techniques supported by your experiments**.

Possible choices:

- Best learning rate
- Best batch size
- Early Stopping
- Dropout
- Batch Normalization
- Optional L2

Do not add a technique only because it is popular.  
Your final configuration must be justified by evidence.

In [23]:
# TODO: Build, compile, train, and evaluate your final optimized model.
#
# final_model = ...
# final_model.compile(...)
# final_history = final_model.fit(...)
# plot_learning_curves(final_history, "Final Optimized Model")
# final_model.evaluate(X_test, y_test)

# Part 8 — Final Experiment Table

| Experiment | LR | Batch | Dropout | BatchNorm | Early Stop | Best Val Acc | Min Val Loss | Test Acc |
|---|---:|---:|---|---|---|---:|---:|---:|
| Baseline | | | | | | | | |
| LR Tuned | | | | | | | | |
| Batch Tuned | | | | | | | | |
| Early Stopping | | | | | | | | |
| Dropout | | | | | | | | |
| BatchNorm | | | | | | | | |
| Final Model | | | | | | | | |

In [24]:
experiment_results = pd.DataFrame(columns=[
    "Experiment", "Learning Rate", "Batch Size",
    "Dropout", "BatchNorm", "Early Stopping",
    "Best Val Accuracy", "Min Val Loss", "Test Accuracy"
])

experiment_results

,Experiment,Learning Rate,Batch Size,Dropout,BatchNorm,Early Stopping,Best Val Accuracy,Min Val Loss,Test Accuracy


# Part 9 — Final Model Analysis

Answer using evidence from your results:

1. Which architecture generalized best: shallow, medium, or deep?
2. Did increasing depth always improve performance?
3. Which model showed the clearest overfitting?
4. Which learning rate produced the most stable training?
5. How did batch size affect the model?
6. Did Early Stopping help?
7. Did Dropout reduce the train/validation gap?
8. Did BatchNorm improve stability or convergence?
9. Which optimization technique had the largest useful effect?
10. Which final model do you recommend, and why?

# Required Final Conclusion

Your conclusion should tell the experimental story:

**Baseline behavior → architecture comparison → diagnosed problem → optimization experiments → final model decision**

A strong conclusion does not say only:

> “Model 3 had the highest accuracy.”

It explains:

> **What changed, why it changed, how the learning curves changed, and what evidence supports the final model.**

# Submission Checklist

- [ ] Dataset description
- [ ] Fixed train / validation / test split
- [ ] Preprocessing without leakage
- [ ] Shallow model
- [ ] Medium model
- [ ] Deep model
- [ ] `model.summary()` and parameter counts
- [ ] Learning curves for all architecture experiments
- [ ] Architecture comparison table
- [ ] Learning-rate tuning
- [ ] Batch-size tuning
- [ ] Early Stopping
- [ ] Dropout
- [ ] Batch Normalization
- [ ] Final optimized model
- [ ] Final experiment table
- [ ] Evidence-based interpretation
- [ ] Final recommendation